# UAV–ISWPT Dataset Generation

This notebook contains only the physics-based, resumable synthetic-data generator.  It does **not** train or inspect any machine-learning model.

**Scientific status of the data:** every input scenario is sampled before the label is computed; every label is produced by minimizing Kang's P1 model with the documented fast inner solver and deterministic 10+5 altitude search.  These are simulation/optimizer labels, not field measurements and not data designed to favor a particular regressor.

The default target is 1000 rows. Generation is resumable through `ds3.npz`; one run adds only the requested chunk.


## Environment

Run the installation line only if the environment is missing a dependency. In Colab, restart is normally unnecessary for these packages.


In [ ]:

# Optional installation:
# %pip install -q numpy pandas scipy cvxpy clarabel scikit-learn matplotlib

import sys, subprocess, json
from pathlib import Path
import numpy as np
import pandas as pd

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)


## Reproducibility rules

- Sample `i` always uses seed `90000+i`.
- Target angles are drawn in `[-70°,70°]`, sorted, and kept at least `15°` apart.
- EHD distances are drawn independently in `[30,200] m`; EHD angles in `[-85°,85°]`.
- The trade-off weight is drawn uniformly in `[0.1,0.9]`.
- Desired power is one quarter of Kang's per-device maximum.
- No row is accepted, rejected, or modified based on ML performance.


## Kang system model and objective

This is the audited source of `kang.py`. The cell writes exactly the code shown here.


In [ ]:
%%writefile kang.py
"""
Faithful implementation of:
  J. Kang, "Joint Design of Transmit Waveform and Altitude for UAV-Enabled ISWPT
  Systems," Electronics 13(21):4237, 2024.
Equation numbers below refer to that paper.
"""
import numpy as np

# ---------------- Table 1 simulation settings ----------------
PT     = 1.0        # transmit sum power (W)
BETA0  = 1.0        # path loss at reference distance
ALPHA0 = 3.5        # path loss exponent, ground links
ALPHA_H= 2.0        # path loss exponent, aerial links (alpha_{pi/2})
A1, B1 = 9.61, 0.16 # LoS-probability params (urban), eq (10)
K0_DB, KH_DB = 5.0, 15.0   # Rician factors in dB at 0 and pi/2, eq (13)
HMIN, HMAX = 50.0, 250.0
N      = 8          # UAV antennas
DA_LC  = 0.5        # d_a / lambda_c
DELTA  = 10.0       # desired beam width (deg)

# derived coefficients, eq (12)
_e = A1 * np.exp(A1 * B1)
A2 = (ALPHA_H - ALPHA0) * (1.0 + _e) / _e
B2 = ALPHA0 - A2 / (1.0 + _e)
# derived coefficients, eq (13)
A3 = K0_DB
B3 = (2.0 / np.pi) * np.log(KH_DB / K0_DB)


def steer(theta_rad):
    """Array steering vector, eq (4)."""
    n = np.arange(N)
    return np.exp(-1j * 2 * np.pi * DA_LC * n * np.sin(theta_rad))


def p_los(psi):
    """LoS probability, eq (10). psi in radians."""
    return 1.0 / (1.0 + A1 * np.exp(-B1 * (np.degrees(psi) - A1)))


def channel_stats(h, r, phi):
    """G_u(h) from eq (15), plus alpha_u and K_u."""
    psi = np.arctan2(h, r)                      # eq (11)
    alpha = A2 * p_los(psi) + B2                # eq (12)
    K = 10.0 ** (0.1 * A3 * np.exp(B3 * psi))   # eq (13), linear
    d = np.sqrt(r ** 2 + h ** 2)                # eq (9)
    a = steer(phi)
    G = BETA0 * d ** (-alpha) * (
        K / (K + 1.0) * np.outer(a, a.conj()) + 1.0 / (K + 1.0) * np.eye(N))
    return G, alpha, K


def g_max(h, r):
    """Max harvested power with MRT beamforming, eq (36)."""
    psi = np.arctan2(h, r)
    alpha = A2 * p_los(psi) + B2
    K = 10.0 ** (0.1 * A3 * np.exp(B3 * psi))
    return BETA0 * (r ** 2 + h ** 2) ** (-alpha / 2.0) * PT * (N * K + 1.0) / (K + 1.0)


def max_desired_power(r, n_grid=400):
    """Solve P4'' by 1-D line search -> max desired power for an EHD."""
    hs = np.linspace(HMIN, HMAX, n_grid)
    return float(np.max([g_max(h, r) for h in hs]))


def angle_grid(res_deg=1.0):
    """Sample angle grid [theta_l], uniform over [-90, 90]."""
    return np.radians(np.arange(-90.0, 90.0 + 1e-9, res_deg))


def desired_pattern(theta_grid, target_angles_rad):
    """chi(theta_l), eq (60): unit gain within DELTA/2 of each target."""
    chi = np.zeros(len(theta_grid))
    half = np.radians(DELTA / 2.0)
    for t in target_angles_rad:
        chi[np.abs(theta_grid - t) <= half + 1e-12] = 1.0
    return chi


def steer_matrix(theta_grid):
    """A[l] = a(theta_l); returns (L, N) complex matrix."""
    return np.array([steer(t) for t in theta_grid])


def beampattern(X, Agrid):
    """P_Rad(theta_l; X) = a^H X a, eq (5), vectorised over the grid."""
    return np.real(np.einsum('li,ij,lj->l', Agrid.conj(), X, Agrid))


def loss_rad(X, Agrid, chi, gamma):
    """L_Rad(X, gamma), eq (6)."""
    return float(np.mean((gamma * chi - beampattern(X, Agrid)) ** 2))


def loss_wpt(X, h, r_list, phi_list, pdes):
    """L_WPT(X, h), eq (16)."""
    v = []
    for r, phi, pd in zip(r_list, phi_list, pdes):
        G, _, _ = channel_stats(h, r, phi)
        v.append((1.0 - np.real(np.trace(G @ X)) / pd) ** 2)
    return float(np.mean(v))


def objective(X, gamma, h, Agrid, chi, r_list, phi_list, pdes, rho):
    """Objective of problem P1."""
    return rho * loss_rad(X, Agrid, chi, gamma) + \
           (1 - rho) * loss_wpt(X, h, r_list, phi_list, pdes)


## Approximate convex inner solver used for bulk labels

This is the audited source of `fast.py`. The cell writes exactly the code shown here.


In [ ]:
%%writefile fast.py
"""
Fast solver for problem P2^h.
The inner problem is a convex quadratic in (X, gamma) over
C = {X Hermitian, X >= 0, [X]_nn = PT/N}.  We use FISTA with a Dykstra
projection onto C, and the closed-form minimiser for gamma at each step.
"""
import numpy as np
from kang import N, PT


def _proj_C(M, iters=25):
    """Dykstra projection onto {X>=0} ∩ {diag(X)=PT/N}."""
    c = PT / N
    X = 0.5 * (M + M.conj().T)
    p = np.zeros_like(X); q = np.zeros_like(X)
    for _ in range(iters):
        Y = X + p
        w, V = np.linalg.eigh(0.5 * (Y + Y.conj().T))
        Y2 = (V * np.maximum(w, 0.0)) @ V.conj().T          # PSD projection
        p = Y - Y2
        Z = Y2 + q
        Z2 = Z.copy()
        np.fill_diagonal(Z2, c)                              # diagonal projection
        q = Z - Z2
        X = Z2
    return X


def solve_fast(Agrid, chi, Glist, pdes, rho, n_iter=120, tol=1e-11, X0=None):
    """Returns (objective, X, gamma)."""
    L = len(chi); U = len(Glist)
    cr = rho / L
    cw = (1.0 - rho) / U
    pd = np.asarray(pdes, float)
    Gs = np.array(Glist)
    chi2 = float(np.sum(chi ** 2))

    Ac = Agrid.conj()

    def pattern(X):
        return np.real(np.einsum('li,ij,lj->l', Ac, X, Agrid))

    def harvest(X):
        return np.real(np.einsum('uij,ji->u', Gs, X))

    def best_gamma(p):
        return float(np.dot(chi, p) / chi2) if chi2 > 0 else 0.0

    def fval(X, g):
        p = pattern(X); q = harvest(X)
        return cr * np.sum((g * chi - p) ** 2) + cw * np.sum((1 - q / pd) ** 2)

    def grad(X, g):
        p = pattern(X); q = harvest(X)
        w = g * chi - p
        Gr = -2 * cr * (Agrid.T @ (w[:, None] * Ac))
        Gw = -2 * cw * np.einsum('u,uij->ij', (1 - q / pd) / pd, Gs)
        return Gr + Gw

    # Lipschitz estimate via power iteration on the Hessian (linear operator)
    R = np.random.default_rng(0).normal(size=(N, N)) + 1j * np.random.default_rng(1).normal(size=(N, N))
    R = 0.5 * (R + R.conj().T); R /= np.linalg.norm(R)
    Lip = 1.0
    for _ in range(30):
        pv = np.real(np.einsum('li,ij,lj->l', Ac, R, Agrid))
        qv = np.real(np.einsum('uij,ji->u', Gs, R))
        HR = 2 * cr * (Agrid.T @ (pv[:, None] * Ac)) + \
             2 * cw * np.einsum('u,uij->ij', qv / pd ** 2, Gs)
        HR = 0.5 * (HR + HR.conj().T)
        nr = np.linalg.norm(HR)
        if nr < 1e-300:
            break
        Lip = nr; R = HR / nr
    step = 1.0 / (Lip * 1.05 + 1e-30)

    X = _proj_C(np.eye(N) * (PT / N)) if X0 is None else _proj_C(X0)
    Y = X.copy(); t = 1.0
    g = best_gamma(pattern(X))
    prev = fval(X, g)
    for k in range(n_iter):
        g = best_gamma(pattern(Y))
        Xn = _proj_C(Y - step * grad(Y, g), iters=5)
        tn = 0.5 * (1 + np.sqrt(1 + 4 * t * t))
        Y = Xn + ((t - 1) / tn) * (Xn - X)
        X, t = Xn, tn
        if k % 25 == 24:
            g = best_gamma(pattern(X)); cur = fval(X, g)
            if abs(prev - cur) < tol * max(1.0, abs(prev)):
                break
            prev = cur
    X = _proj_C(X, iters=40)
    g = best_gamma(pattern(X))
    return fval(X, g), X, g


## Resumable scenario sampler and label generator

This is the audited source of `gen3.py`. The cell writes exactly the code shown here.


In [ ]:
%%writefile gen3.py
import numpy as np, os, sys, time
from kang import *
from fast import solve_fast

TARGET = 1000
F = "../data/ds3.npz"
CHUNK = int(sys.argv[1]) if len(sys.argv) > 1 else 130
TH = angle_grid(1.0)
AG = steer_matrix(TH)


def sample(seed):
    g = np.random.default_rng(seed)
    while True:
        tg = np.sort(g.uniform(-70, 70, 3))
        if np.min(np.diff(tg)) >= 15:
            break
    r = g.uniform(30, 200, 2)
    phi = g.uniform(-85, 85, 2)
    rho = g.uniform(0.1, 0.9)
    return tg, r, phi, rho


def solve_instance(tg, r, phi, rho, n_coarse=10, n_fine=5):
    chi = desired_pattern(TH, np.radians(tg))
    pdes = [0.25 * max_desired_power(x) for x in r]
    pr = np.radians(phi)
    ns = 0

    def f(h, X0):
        Gl = [channel_stats(h, rr, pp)[0] for rr, pp in zip(r, pr)]
        return solve_fast(AG, chi, Gl, pdes, rho, X0=X0)

    hs = np.linspace(HMIN, HMAX, n_coarse)
    vals = []; X0 = None
    for h in hs:
        v, X0, _ = f(h, X0); ns += 1; vals.append(v)
    i = int(np.argmin(vals)); bv, bh = vals[i], hs[i]
    lo, hi = hs[max(i-1, 0)], hs[min(i+1, n_coarse-1)]
    for h in np.linspace(lo, hi, n_fine):
        v, X0, _ = f(h, X0); ns += 1
        if v < bv: bv, bh = v, h
    return float(bh), float(bv), ns


if os.path.exists(F):
    d = np.load(F)
    X = list(d["X"]); yH = list(d["yH"]); yO = list(d["yO"])
    done = int(d["done"]); ns = int(d["ns"]); tt = float(d["tt"])
else:
    X, yH, yO = [], [], []; done = ns = 0; tt = 0.0

t0 = time.time(); end = min(done + CHUNK, TARGET)
for i in range(done, end):
    tg, r, phi, rho = sample(90000 + i)
    h, o, k = solve_instance(tg, r, phi, rho); ns += k
    X.append([tg[0], tg[1], tg[2], r[0], r[1], phi[0], phi[1], rho])
    yH.append(h); yO.append(o)
tt += time.time() - t0
np.savez(F, X=np.array(X), yH=np.array(yH), yO=np.array(yO),
         done=end, ns=ns, tt=tt)
print(f"{end}/{TARGET}  t={tt:.0f}s  chunk={time.time()-t0:.0f}s  "
      f"solves={ns}  ms/solve={tt/max(ns,1)*1000:.0f}", flush=True)


## Generate one resumable chunk

Set `RUN_ONE_CHUNK=True` and execute this cell. Re-run it until `done` reaches the target. A chunk of 50 is convenient for Jupyter/Colab. To study 4000 samples, change `TARGET = 1000` to `TARGET = 4000` in the visible `gen3.py` cell above and rerun that cell first.


In [ ]:

RUN_ONE_CHUNK = False   # change to True when you intentionally want new labels
CHUNK = 50

if RUN_ONE_CHUNK:
    subprocess.run([sys.executable, "gen3.py", str(CHUNK)], check=True)
else:
    print("Generation is paused. Set RUN_ONE_CHUNK=True to add one chunk.")


## Inspect the dataset without training a model


In [ ]:

dataset_path = Path("../data/ds3.npz")
if not dataset_path.exists():
    print("No ds3.npz yet. Generate at least one chunk above.")
else:
    d = np.load(dataset_path)
    X, y_h, y_obj = d["X"], d["yH"], d["yO"]
    print("rows, features:", X.shape)
    print("finite values:", bool(np.isfinite(X).all() and np.isfinite(y_h).all() and np.isfinite(y_obj).all()))
    print("duplicate feature rows:", len(X) - len(np.unique(X, axis=0)))
    print("altitude range:", float(y_h.min()), "to", float(y_h.max()), "m")
    print("Hmin count:", int(np.sum(np.isclose(y_h, 50.0))))
    print("Hmax count:", int(np.sum(np.isclose(y_h, 250.0))))
    print("optimizer solves:", int(d["ns"]), "elapsed seconds:", float(d["tt"]))


## Export the human-readable CSV


In [ ]:

if dataset_path.exists():
    d = np.load(dataset_path)
    columns = ["theta1", "theta2", "theta3", "r1", "r2", "phi1", "phi2", "rho"]
    frame = pd.DataFrame(d["X"], columns=columns)
    frame["optimal_altitude_m"] = d["yH"]
    frame["stored_objective"] = d["yO"]
    frame.to_csv("../data/dataset.csv", index=False)
    display(frame.head())
    print("wrote dataset.csv with", len(frame), "rows")


## Handoff to the analysis notebook

Keep `ds3.npz` beside `02_Complete_ML_Analysis.ipynb`.  The second notebook treats it as fixed input and never regenerates or filters rows based on the model results.
